Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd


query = """
select 
coalesce(policy_renewed_dt_helper,call_connect_dt) as dt,
CASE WHEN day_of_week(coalesce(policy_renewed_dt_helper,call_connect_dt)) = 6 THEN date_add('day', 2, date(coalesce(policy_renewed_dt_helper,call_connect_dt))) ELSE date_add('day', 1, date(coalesce(policy_renewed_dt_helper,call_connect_dt))) END AS Audit_Date,
sum(case when filter_status = 'd_1_renewed_conversations' then 1 else 0 end) as renewed_conversations_count,
sum(case when filter_status = 'd_1_non_renewed_conversations' then 1 else 0 end) as non_renewed_conversations_count,
count (DISTINCT case when filter_status = 'd_1_renewed_conversations' then policy_id end ) as renewed_policy_id_count 
from(
WITH L0_disposition AS (
    SELECT
        da.ticket_uuid,
        max(da.question_reference_id) AS question_reference_id,
        max(o.text) AS text
        -- , try_cast(from_iso8601_timestamp(da.created_at) + interval '330' minute as timestamp) as disposition_created_at
    FROM crm_tickets.disposition_answers da
    LEFT JOIN crm_tickets.options AS o
        ON split_part(da.option_reference_ids, '"', 2) = o.reference_id
    WHERE da.question_reference_id IN (
        '3082565b-cc22-47f6-8f3d-fe21a6b2ba6c',
        'babcfbfb-efb5-4528-b22e-55ec5bf0c6ba',
        '465a45b1-4250-49af-8f4e-c8e2564a5d71'
    )
    GROUP BY 1
),
when_renew AS (
    SELECT
        da.ticket_uuid,
        max(da.question_reference_id) AS question_reference_id,
        max(o.text) AS text
        -- , try_cast(from_iso8601_timestamp(da.created_at) + interval '330' minute as timestamp) as disposition_created_at
    FROM crm_tickets.disposition_answers da
    LEFT JOIN crm_tickets.options AS o
        ON split_part(da.option_reference_ids, '"', 2) = o.reference_id
    WHERE da.question_reference_id IN ('c19b30ff-22f1-423f-9bc2-77b04eb3d29d')
    GROUP BY 1
),
why_followup AS (
    SELECT
        da.ticket_uuid,
        max(da.question_reference_id) AS question_reference_id,
        max(o.text) AS text
        -- , try_cast(from_iso8601_timestamp(da.created_at) + interval '330' minute as timestamp) as disposition_created_at
    FROM crm_tickets.disposition_answers da
    LEFT JOIN crm_tickets.options AS o
        ON split_part(da.option_reference_ids, '"', 2) = o.reference_id
    WHERE da.question_reference_id IN ('af1e0b77-f9b5-4069-bf87-83271f16ea6a')
    GROUP BY 1
),
what_done AS (
    SELECT
        da.ticket_uuid,
        max(da.question_reference_id) AS question_reference_id,
        max(o.text) AS text
        -- , try_cast(from_iso8601_timestamp(da.created_at) + interval '330' minute as timestamp) as disposition_created_at
    FROM crm_tickets.disposition_answers da
    LEFT JOIN crm_tickets.options AS o
        ON split_part(da.option_reference_ids, '"', 2) = o.reference_id
    WHERE da.question_reference_id IN ('2323a7b7-62d2-469a-967b-4c4f531c6a6b')
    GROUP BY 1
),
connected_calls AS (
    SELECT
        cl.call_group_id AS uuid,
        ce.call_id,
        replace(u.name, '.', ' ') AS name,
        CASE WHEN ij.call_id IS NOT NULL THEN 'CONNECTED' ELSE 'Other' END AS call_connection_status,
        max(CASE
                WHEN cast(json_extract_scalar(ce.attributes, '$.callStatus') AS varchar) = 'CONNECTED'
                THEN try_cast(parse_datetime(substring(json_extract_scalar(ce.attributes, '$.callConnectedTime'), 1, 19), 'yyyy/MM/dd HH:mm:ss') AS timestamp)
                ELSE NULL
            END) AS call_connected_ist,
        max(CASE
                WHEN event_type = 'CALL_SUMMARY'
                THEN try_cast(split_part(json_extract_scalar(ce.attributes, '$.talkTime'), ':', 1) AS int) * 3600
                   + try_cast(split_part(json_extract_scalar(ce.attributes, '$.talkTime'), ':', 2) AS int) * 60
                   + try_cast(split_part(json_extract_scalar(ce.attributes, '$.talkTime'), ':', 3) AS int)
                ELSE NULL
            END) AS call_duration,
        max(CASE WHEN event_type = 'CALL_SUMMARY' THEN json_extract_scalar(ce.attributes, '$.numAttempts') ELSE NULL END) AS AttemptNum,
        max(CASE WHEN event_type = 'CALL_SUMMARY' THEN replace(split_part(json_extract_scalar(ce.attributes, '$.userId'), '@', 1), '.', ' ') ELSE NULL END) AS Agent_name,
        max(CASE WHEN event_type = 'CALL_SUMMARY' THEN json_extract_scalar(ce.attributes, '$.recordingUrl') ELSE NULL END) AS recordingUrl
    FROM graham_bell_service.call_events ce
    LEFT JOIN (
        SELECT DISTINCT call_id
        FROM graham_bell_service.call_events
        WHERE event_type = 'DISPOSITION_SUBMITTED'
    ) ij ON ij.call_id = ce.call_id
    LEFT JOIN graham_bell_service.call_logs AS cl ON ce.call_id = cl.call_id
    LEFT JOIN mjolnir_service.users AS u ON cl.navi_agent_id = u.account_id
    WHERE cl.tenant_identifier = 'GI_RENEWAL_CALLING'
    GROUP BY 1, 2, 3, 4
),
attempt_time AS (
    SELECT
        ticket_uuid,
        min_by(call_initiation_time, attempt_number) AS first_initiation_time
    FROM (
        SELECT
            cl.call_group_id AS ticket_uuid,
            CASE WHEN ce.event_type = 'CALL_SUMMARY' THEN try_cast(json_extract_scalar(ce.attributes, '$.numAttempts') AS int) ELSE NULL END AS attempt_number,
            CASE WHEN ce.event_type = 'CALL_SUMMARY'
                 THEN try_cast(date_format(date_parse(substring(json_extract_scalar(ce.attributes, '$.callInitiationTime'), 1, 19), '%Y/%m/%d %H:%i:%s'), '%Y-%m-%d %H:%i:%s') AS timestamp)
            END AS call_initiation_time
        FROM graham_bell_service.call_events ce
        LEFT JOIN graham_bell_service.call_logs AS cl ON cl.call_id = ce.call_id
        WHERE ce.event_type = 'CALL_SUMMARY'
          AND cl.tenant_identifier = 'GI_RENEWAL_CALLING'
    )
    GROUP BY 1
),

main_table AS (
    SELECT
        try_cast(from_iso8601_timestamp(t.created_at) + interval '330' minute AS timestamp) AS ticket_creation_ist,
        t.uuid AS crm_ticket_id,
        t.priority,
        coalesce(cc.call_connection_status, 'Other') AS call_connection_status,
        t.reporter_id AS navi_customer_id,
        date(try_cast(cc.call_connected_ist AS timestamp)) AS call_connect_dt,
        coalesce(cc.name, cc.Agent_name) AS Agent_name,
        cc.call_duration,
        json_extract_scalar(t.meta_data, '$.policyId') AS policy_id,
        json_extract_scalar(t.meta_data, '$.renewalType') AS renewal_type,
        att.first_initiation_time,
        cast(crd.policy_renewal_ts as date) as policy_renewed_dt,
        date(ps.policy_issued_created_at) + interval '365' day as policy_renewal_dt,
        COALESCE(cast(crd.policy_renewal_ts as date),MAX(cast(crd.policy_renewal_ts as date)) OVER (PARTITION BY policy_id)) AS policy_renewed_dt_helper,
        (date(ps.policy_issued_created_at) + interval '365' day) - interval '62' day as call_window_starts_from,
        l0d.text AS L0_disposition
    FROM crm_tickets.tickets t
    LEFT JOIN L0_disposition AS l0d ON t.uuid = l0d.ticket_uuid
    LEFT JOIN connected_calls AS cc ON t.uuid = cc.uuid
    LEFT JOIN why_followup AS fl ON t.uuid = fl.ticket_uuid
    LEFT JOIN when_renew AS renew ON t.uuid = renew.ticket_uuid
    LEFT JOIN what_done AS wd ON t.uuid = wd.ticket_uuid
    LEFT JOIN attempt_time AS att ON att.ticket_uuid = t.uuid
    LEFT JOIN corp_analytics.gi_renewal_calling_ticket_datamodel_tbl crd on crd.policy_id = json_extract_scalar(t.meta_data, '$.policyId')
    LEFT JOIN corp_analytics.gi_policy_sales_fct_view AS ps ON json_extract_scalar(t.meta_data, '$.policyId') = ps.base_policy_id
    WHERE t.ticket_group = 'GI_RENEWAL_CALLING'
),

filter_tagging as (
select 
*, 
case when policy_renewed_dt_helper is not null and call_connect_dt >= call_window_starts_from then 'd_1_renewed_conversations'
     when  policy_renewed_dt_helper is null then 'd_1_non_renewed_conversations' 
     else null
end as filter_status
from main_table
where call_connection_status = 'CONNECTED'
)/*,

allocation_tagging as (
select 
*, 
case when filter_status = 'd_1_renewed_conversations' and policy_renewed_dt_helper = current_date - interval '1' day then 'dminus1'
     when filter_status = 'd_1_renewed_conversations' and policy_renewed_dt_helper = current_date - interval '2' day then 'dminus2'
     when filter_status = 'd_1_non_renewed_conversations' and call_connect_dt = current_date - interval '1' day then 'dminus1'
     when filter_status = 'd_1_non_renewed_conversations' and call_connect_dt = current_date - interval '2' day then 'dminus2'
end as allocation_type
from filter_tagging
)*/


select DISTINCT 
crm_ticket_id,
Agent_name,
call_duration,
call_connect_dt,
Null AS call_connect_timestamp,
navi_customer_id,
null as policy_renewed_dt,
L0_disposition,
filter_status,
--allocation_type,
policy_renewed_dt_helper,
policy_id
FROM filter_tagging
WHERE filter_status = 'd_1_renewed_conversations'
and policy_renewed_dt_helper between current_date - interval '90' day and current_date - interval '1' day

union


select DISTINCT 
crm_ticket_id,
Agent_name,
call_duration,
call_connect_dt,
Null AS call_connect_timestamp,
navi_customer_id,
null as policy_renewed_dt,
L0_disposition,
filter_status,
--allocation_type,
policy_renewed_dt_helper,
policy_id
FROM filter_tagging
WHERE filter_status = 'd_1_non_renewed_conversations'
and call_connect_dt between current_date - interval '90' day and current_date - interval '1' day
)
group by 1,2
order by 1 desc
"""

df  = get_sql_df(query)
if not df.empty:
    display(df)

In [ ]:
# DBTITLE 1,Cell 3
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1FoHpHPMlw5HclJygb48_cLbHT6Lu5rV7be23DweUerI/edit?pli=1&gid=527213770#gid=527213770"
worksheet_name = "Renewal Connected Call Count 2"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

MAX_COL = 5  # columns A–E only; do NOT touch F

combined_df = pd.concat([df], ignore_index=True)
combined_df = combined_df.iloc[:, :MAX_COL]

# --- clear ONLY A:E (never use clear_sheets here) ---
worksheet = gs_reader.get_sheet(worksheet_name)

existing_rows = len(worksheet.col_values(1))          # rows with data in col A
new_rows = len(combined_df) + 1                       # +1 for header row
rows_to_clear = max(existing_rows, new_rows)

worksheet.batch_clear([f"A1:E{rows_to_clear}"])       # clears A–E only; F is untouched

# --- write new data into A:E ---
gs_reader.set_df_in_sheets(
    worksheet_name,
    combined_df,
    include_headers=True,
    row=1,
    col=1,
)